# Computing the Canonical Model of $X_0(163)$

This script computes the defining equations for the modular curve $X_0(163)$ of genus $g=13$. 
Since $X_0(163)$ is non-hyperelliptic, its canonical embedding into $\mathbb{P}^{12}$ is determined by quadratic relations (by Petri's Theorem).

**Key Features:**
* **Basis Construction:** Computes an integral basis for the space of cusp forms $S_2(\Gamma_0(163))$ using **LLL reduction**. This ensures that the basis elements have minimal integer coefficients in their $q$-expansions.
* **Canonical Embedding:** Solves for linear dependencies among the quadratic monomials $x_i x_j$ to generate the defining ideal of the curve.
* **Precision Management:** Automatically handles $q$-series precision to rigorously establish linear independence.

In [ ]:
from sage.all import *

class CanonicalModularCurve:
    """
    Computes the defining equations of the modular curve X_0(N).
    Reproduces the LLL-reduced basis logic from the original Galbraith method
    to ensure consistent, minimal coefficients.
    """
    def __init__(self, N, prec=500):
        self.N = N
        self.prec = prec
        
        self.S2 = CuspForms(N, 2)
        self.genus = self.S2.dimension()
        
        self.R = LaurentSeriesRing(QQ, 'q', default_prec=self.prec)
        self.q = self.R.gen()
        
        self.basis = [] 
        self.PolyR = PolynomialRing(QQ, 'x', self.genus)
        self.vars = self.PolyR.gens()

    def compute_reduced_basis(self):
        """
        Computes LLL-reduced integral basis using a deep coefficient check
        to match the original output style.
        """
        print(f"Computing LLL-reduced basis (Genus {self.genus}, Prec q^{self.prec})...")
        initial_basis = self.S2.integral_basis()
        
        if not initial_basis: return []

        # [Updated] Use a deeper coefficient window for LLL to match
        # the original output style and obtain shorter/nicer basis elements.
        # Default: genus * 10 + 50 (about 180 terms for N=163).
        lll_depth = self.genus * 10 + 50
        
        # Precision guard
        if lll_depth > self.prec:
            lll_depth = self.prec - 10

        # Matrix for LLL
        mat_data = []
        for f in initial_basis:
            f_q = f.q_expansion(lll_depth + 10)
            row = [f_q[n] for n in range(1, lll_depth)] # q^1 through q^lll_depth
            mat_data.append(row)

        coeff_matrix = Matrix(ZZ, mat_data)
        
        # Apply LLL
        _, U = coeff_matrix.LLL(transformation=True)
        
        reduced_basis = []
        for row in U:
            f_combined = sum(row[j] * initial_basis[j] for j in range(self.genus))
            
            # Final expansion to full precision (500)
            f_series = self.R(f_combined.q_expansion(self.prec))
            
            # Normalize sign
            val = f_series.valuation()
            if val < self.prec and f_series[val] < 0:
                f_series = -f_series
            
            reduced_basis.append(f_series)
            
        self.basis = reduced_basis
        return self.basis

    def find_equations(self):
        if not self.basis: self.compute_reduced_basis()
            
        print(f"Searching for quadratic relations...")
        
        monomials_q = []
        monomials_alg = []
        
        for i in range(self.genus):
            for j in range(i, self.genus):
                monomials_q.append(self.basis[i] * self.basis[j])
                monomials_alg.append(self.vars[i] * self.vars[j])
        
        # Kernel Check Length
        check_len = len(monomials_q) + 20
        min_val = min(f.valuation() for f in monomials_q)
        
        if min_val + check_len > self.prec:
             check_len = self.prec - min_val - 1
        
        mat_data = [[f[n] for n in range(min_val, min_val + check_len)] for f in monomials_q]
        M = Matrix(QQ, mat_data)
        
        kernel = M.kernel().basis()
        
        equations = []
        for vec in kernel:
            poly = sum(c * m for c, m in zip(vec, monomials_alg))
            
            coeffs = poly.coefficients()
            if not coeffs: continue
            
            # Normalize
            denom = lcm([c.denominator() for c in coeffs])
            poly_int = poly * denom
            
            # GCD reduction for cleaner look
            common_factor = gcd(poly_int.coefficients())
            if common_factor > 1:
                poly_int = poly_int // common_factor

            if poly_int.coefficients()[-1] < 0: 
                poly_int = -poly_int
            
            equations.append(poly_int)
            
        print(f"Found {len(equations)} defining equations.")
        return equations

# ===================================================
# Execution
# ===================================================

curve = CanonicalModularCurve(163, prec=1000)
eq_list = curve.find_equations()

print("\n" + "="*60)
print(f" [Basis Elements: x0 ... x{curve.genus-1}] (First few terms)")
print("="*60)
# Show only the first 3 basis elements and terms up to q^20 for preview.
for i, f in enumerate(curve.basis):
    print(f"x{i} = {f + O(curve.q**20)}")

print("\n" + "="*60)
print(f" Defining Equations for X_0(163) (Total: {len(eq_list)})")
print("="*60)
for i, eq in enumerate(eq_list):
    print(f"[{i+1}] {eq}")

### Plane Model and Genus Verification

We define modular functions $X = x_1/x_{11}$ and $Y = x_6/x_{11}$ using the canonical basis. 

By analyzing the linear dependencies of their $q$-expansions, we compute the defining polynomial $F(X, Y) = 0$. 

Finally, we calculate the geometric genus of this plane curve to verify that it is birational to $X_0(163)$ (i.e., $g=13$).

In [ ]:
def find_bivariate_relation(X_series, Y_series, max_deg=10):
    """
    Finds the irreducible polynomial F(X, Y) such that F(X_series, Y_series) = O(q^N).
    Uses linear algebra (kernel of the coefficient matrix).
    """
    print(f"Searching for relation between X and Y (Max Degree {max_deg})...")
    
    # Define Polynomial Ring
    PolyR = PolynomialRing(QQ, ['X', 'Y'])
    X, Y = PolyR.gens()
    
    # Precision safety: We need enough q-coefficients to solve the linear system.
    # System size ~ (d+1)(d+2)/2. We use the available precision of the series.
    prec_limit = min(X_series.precision_absolute(), Y_series.precision_absolute())
    
    for d in range(1, max_deg + 1):
        monomials_alg = []
        monomials_q = []
        
        # Generate monomials of total degree <= d
        for i in range(d + 1):
            for j in range(d - i + 1):
                monomials_alg.append(X**i * Y**j)
                monomials_q.append(X_series**i * Y_series**j)
        
        # Construct Matrix: Rows = q-coefficients, Cols = Monomials
        # We check enough coefficients to overdetermine the system
        num_cols = len(monomials_q)
        check_len = num_cols + 20 
        
        # Determine valid range for coefficients
        min_val = min(m.valuation() for m in monomials_q)
        if min_val + check_len > prec_limit:
            check_len = prec_limit - min_val - 1
            if check_len <= num_cols:
                print(f"  [Warning] Insufficient precision at degree {d}. Stopping.")
                return None

        mat_data = []
        for m in monomials_q:
            mat_data.append([m[n] for n in range(min_val, min_val + check_len)])
            
        M = Matrix(QQ, mat_data)
        
        # Compute Kernel (Linear Dependencies)
        ker = M.kernel()
        
        if ker.dimension() > 0:
            vec = ker.basis()[0]
            
            # Reconstruct Polynomial
            F = sum(c * m for c, m in zip(vec, monomials_alg))
            
            # Normalize (Integer coefficients, positive leading term)
            coeffs = F.coefficients()
            denom = lcm([c.denominator() for c in coeffs])
            F_int = F * denom
            
            # Divide by common content to make primitive
            cont = gcd(F_int.coefficients())
            F_final = F_int // cont
            
            if F_final.coefficients()[-1] < 0:
                F_final = -F_final
                
            return F_final
            
    print("No relation found within max_deg.")
    return None

# ===================================================
# 1. Define Modular Functions X and Y
# ===================================================
# Note: Ensure indices exist in curve.basis (len >= 12 for N=163)
# Using indices 1, 6, 11 as requested.
# If these indices are arbitrary, the resulting curve might be highly singular.
try:
    # Use basis from the previous step
    x_basis = curve.basis 
    X_mod = x_basis[1] / x_basis[11]
    Y_mod = x_basis[6] / x_basis[11]
    
    print("Defined modular functions:")
    print(f" X = x1/x11 = {X_mod + O(curve.q**5)}")
    print(f" Y = x6/x11 = {Y_mod + O(curve.q**5)}")

except IndexError:
    print("Error: Basis list is too short. Please check the genus and basis size.")
    raise

# ===================================================
# 2. Compute Algebraic Relation F(X, Y) = 0
# ===================================================

# Search for relation (degree can be high for projections, e.g., 10+)
F_XY = find_bivariate_relation(X_mod, Y_mod, max_deg=20)

if F_XY:
    print("\n" + "="*60)
    print(" [Plane Curve Model]")
    print(f" Equation: {F_XY} = 0")
    print(f" Total Degree: {F_XY.total_degree()}")
    print(f" Number of Terms: {len(F_XY.monomials())}")
    print("="*60)

    # ===================================================
    # 3. Compute Genus of the Plane Curve
    # ===================================================
    print("\nComputing geometric genus of the plane curve...")
    
    # Define Affine Space and Curve
    A2 = AffineSpace(QQ, 2, 'X,Y')
    C_plane = Curve([F_XY], A2)
    
    # Calculate Geometric Genus
    # Note: simple 'C_plane.genus()' computes the geometric genus (normalization)
    # which is the correct modular genus, even if the plane model is singular.
    g_geom = C_plane.geometric_genus()
    
    print(f"Geometric Genus: {g_geom}")
    
    # Verification
    if g_geom == curve.genus:
        print(">> SUCCESS: Matches the genus of X_0(163).")
    else:
        print(f">> WARNING: Mismatch. (Expected {curve.genus}, got {g_geom})")
        print("   The map (X,Y) might not be birational to X_0(N).")
else:
    print("Failed to find a defining equation.")

## Explicit Construction of a Rational Heegner Point

In this section, we determine the explicit coordinates of a Heegner point on the curve model $F(X, Y) = 0$.

We consider the imaginary quadratic field $K = \mathbb{Q}(\sqrt{-163})$. Since the class number is $h(-163)=1$, the Heegner point associated with the maximal order $\mathcal{O}_K$ is defined over $\mathbb{Q}$. The corresponding CM point is:
$$\tau = \frac{-163 + \sqrt{-163}}{326}$$

**Methodology:**
1.  **High-Precision Evaluation:** We evaluate the modular parameterizations $X(\tau)$ and $Y(\tau)$ using the $q$-expansions computed to precision $O(q^{1000})$.
2.  **Error Analysis:** With the series truncated at degree $N=1000$ and $|q| \approx 0.782$, the truncation error is significantly reduced to:
    $$|q|^{1000} \approx 10^{-107}$$
    This provides numerical accuracy to approximately **107 decimal places**, ensuring an extremely high degree of confidence in identifying the rational coordinates.
3.  **Rational Identification:** Using continued fractions, we recover the exact rational coordinates from the numerical approximations.
4.  **Algebraic Verification:** We rigorously verify that the identified point $(X, Y) \in \mathbb{Q}^2$ satisfies $F(X, Y) = 0$ exactly.

In [ ]:
from fractions import Fraction

def verify_heegner_point_values():
    print("=" * 70)
    print(" Explicit Construction of the Heegner Point (D = -163)")
    print("=" * 70)

    # 1. Setup: High-Precision Environment
    # We use 2000 bits of working precision.
    # With series computed up to q^1000, the theoretical error is |q|^1000 ~ 1e-107.
    CC_huge = ComplexField(2000)
    val_sqrt_163 = CC_huge(163).sqrt()
    tau = (-163 + I * val_sqrt_163) / 326
    q_val = (2 * CC_huge.pi() * I * tau).exp()

    print(f" [1] CM Point Setup")
    print(f"     tau : {tau:.10f}")
    print(f"     |q| : {abs(q_val):.10f} (Convergence rate)")

    # 2. Evaluation of Modular Series
    def eval_series(f, val):
        # Convert Laurent series to polynomial for efficient high-precision evaluation
        return f.power_series().polynomial()(val)

    try:
        # Check if the basis has enough precision (optional safety check)
        # Assuming 'curve.basis' was generated with prec=1000
        val_1 = eval_series(curve.basis[1], q_val)
        val_6 = eval_series(curve.basis[6], q_val)
        val_11 = eval_series(curve.basis[11], q_val)
    except (NameError, IndexError):
        print("Error: Basis elements from the canonical embedding are missing.")
        return

    # Compute coordinate values
    X_num = (val_1 / val_11).real()
    Y_num = (val_6 / val_11).real()

    print(f"\n [2] Numerical Evaluation (Precision ~ 107 decimal places)")
    print(f"     X (approx) = {X_num:.20f}...")
    print(f"     Y (approx) = {Y_num:.20f}...")

    # 3. Identification of Rational Coordinates
    # Using continued fractions to filter out machine noise.
    # Given the high precision, we can be very confident in these rationals.
    
    X_simple = Fraction(float(X_num)).limit_denominator(100000) # Increased limit safely
    Y_simple = Fraction(float(Y_num)).limit_denominator(100000)

    X_rat = QQ(X_simple)
    Y_rat = QQ(Y_simple)

    print(f"\n [3] Rational Identification (Continued Fraction method)")
    print(f"     Candidate X = {X_rat}")
    print(f"     Candidate Y = {Y_rat}")

    # 4. Algebraic Verification
    # Explicitly check if F(X, Y) == 0 using exact rational arithmetic.
    if 'F_XY' in globals():
        print(f"\n [4] Algebraic Verification on the Curve F(X, Y) = 0")
        val_exact = F_XY(X=X_rat, Y=Y_rat)
        
        print(f"     Residual F({X_rat}, {Y_rat}) = {val_exact}")
        
        if val_exact == 0:
            print("-" * 70)
            print("     >> SUCCESS: The point lies EXACTLY on the curve.")
            print(f"     >> Confirmed Heegner Point: P = ({X_rat}, {Y_rat})")
            print("-" * 70)
        else:
            print(f"     >> FAILURE: The point is not a root. (Residual: {float(val_exact)})")
    else:
        print("\n [Warning] Defining polynomial 'F_XY' not found. Verification skipped.")

# Execute the verification
verify_heegner_point_values()

### Analytic Construction of the Cyclic 163-Isogeny

This script implements an analytic method to construct the cyclic isogeny for $N=163$, bypassing the computational intractability of the algebraic approach.

**Key Computations:**
1.  **High-Precision Evaluation:** Evaluates the modular forms $E_2, E_4, E_6$ at the Heegner point $\tau = \frac{-163 + \sqrt{-163}}{326}$ using **4000 bits of precision** (approx. 1200 decimal digits).
2.  **Invariant Calculation:** Computes the numerical values of the Weierstrass invariants $(a_4, a_6)$ for the domain curve $E$ and $(a_4', a_6')$ for the codomain curve $E'$.
3.  **Rational Reconstruction:** Recovers the **exact rational coefficients** from the high-precision floating-point results.
4.  **Verification:** Confirms that the $j$-invariants of both reconstructed curves match the theoretical value $j(-163) = -640320^3$.
5.  **Twist Analysis:** Identifies the constructed curves as quadratic twists of the minimal model **26569.a1**. It verifies the relation $D' = -163 \times D$ between the twist factors, confirming that the isogeny is induced by the multiplication-by-$\sqrt{-163}$ endomorphism.

In [ ]:

def compute_full_isogeny_163():
    print("=" * 70)
    print(" Analytic Construction of Cyclic Isogeny (E -> E') for N=163")
    print("=" * 70)

    # 1. Setup High Precision Environment (4000 bits)
    PRECISION_BITS = 4000
    R = RealField(PRECISION_BITS)
    N = 163
    
    # 2. Define q-parameters
    # tau = (-163 + sqrt(-163)) / 326
    pi_val = R.pi()
    sqrt_N = R(N).sqrt()
    
    q_tau = -(-pi_val / sqrt_N).exp()
    q_Ntau = -(-pi_val * sqrt_N).exp()

    print(f" [1] Numerical Setup")
    print(f"     Precision : {PRECISION_BITS} bits")
    print(f"     q (Domain): {float(q_tau):.6f}")
    print(f"     q (Codom.): {float(q_Ntau):.6e}")

    # 3. Optimized Modular Evaluation Function
    def eval_modular_forms(q_val):
        q_mag = abs(q_val)
        if q_mag == 0: return R(1), R(1), R(1)
        
        iter_limit = int(-PRECISION_BITS * 0.693 / q_mag.log()) + 50
        iter_limit = min(iter_limit, 50000)
        
        s1, s3, s5 = R(0), R(0), R(0)
        curr_q = q_val
        
        for n in range(1, iter_limit):
            term = curr_q / (1 - curr_q)
            n_R = R(n)
            n2 = n_R * n_R
            
            s1 += n_R * term
            s3 += (n2 * n_R) * term
            s5 += (n2 * n2 * n_R) * term
            
            curr_q *= q_val
            if abs(curr_q) < R(2)**(-PRECISION_BITS - 10): break
                
        return (1 - 24*s1, 1 + 240*s3, 1 - 504*s5)

    # 4. Compute Invariants
    print("\n [2] Computing Invariants (Analytic)...")
    E2_tau, E4_tau, E6_tau = eval_modular_forms(q_tau)
    E2_Ntau, E4_Ntau, E6_Ntau = eval_modular_forms(q_Ntau)

    S = E2_tau - N * E2_Ntau # Scaling Factor
    
    # Domain (E) and Codomain (E') Invariants
    val_a4 = -E4_tau / (48 * S**2)
    val_a6 = E6_tau / (864 * S**3)
    val_a4_p = -E4_Ntau / (48 * S**2)
    val_a6_p = E6_Ntau / (864 * S**3)

    # 5. Rational Reconstruction
    print("\n [3] Rational Reconstruction")
    
    def recover_rational(val):
        exact_q = val.exact_rational()
        frac = Fraction(int(exact_q.numerator()), int(exact_q.denominator()))
        return frac.limit_denominator(10**25)

    rec_a4 = recover_rational(val_a4)
    rec_a6 = recover_rational(val_a6)
    rec_a4_p = recover_rational(val_a4_p)
    rec_a6_p = recover_rational(val_a6_p)

    print(f"     E (a4) : {rec_a4}")
    print(f"     E (a6) : {rec_a6}")
    print(f"     E' (a4): {rec_a4_p}")
    print(f"     E' (a6): {rec_a6_p}")

    # 6. Twist Factor Analysis
    print("\n [4] Twist & Conductor Analysis")
    
    # Define Curves in Sage
    E_domain = EllipticCurve([QQ(rec_a4), QQ(rec_a6)])
    E_codomain = EllipticCurve([QQ(rec_a4_p), QQ(rec_a6_p)])

    # Compute Minimal Twist and Conductors
    E_min, D = E_domain.minimal_quadratic_twist()
    Ep_min, D_p = E_codomain.minimal_quadratic_twist()
    
    cond = E_min.conductor()
    
    print(f"     Minimal Conductor : {cond} (Expected: 163^2 = 26569)")
    print(f"     Twist Factor D    : {D}")
    print(f"     Twist Factor D'   : {D_p}")

    # Verify the relationship D' = -163 * D
    print("\n [5] Verifying CM Twist Relationship")
    
    expected_Dp = -163 * D
    print(f"     Expected D'       : {expected_Dp} (-163 * D)")
    
    if D_p == expected_Dp:
        print("     >> SUCCESS: D' = -163 * D confirmed.")
        print("     >> The isogeny is compatible with the CM by sqrt(-163).")
    else:
        print(f"     >> FAILURE: Twist factors do not match the expected relation.")

    # 7. Verification against Paper
    target_a4 = Fraction("-543605/75481344")
    if rec_a4 == target_a4 and D_p == expected_Dp:
        print("\n" + "="*70)
        print(" FINAL RESULT: All computed values match the published paper [Jeon-Kwon].")
        print("="*70)

# Execute
compute_full_isogeny_163()